# Detector Variation Samples

In [ ]:
%load_ext autoreload
%autoreload 2

#print all output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
from os import path
import sys
import uproot
from tqdm import tqdm
import datetime

# local imports
# sys.path.append('../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.selection_definitions import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.makedf.util import *
from pyanalib.split_df_helpers import *
from pyanalib.stat_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.variable_calculator import get_cc1p0pi_tki
from pyanalib.pandas_helpers import pad_column_name
from makedf.constants import *

plt.style.use("presentation.mplstyle")
cmap = mpl.cm.viridis
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredText
from matplotlib.offsetbox import AnchoredOffsetbox, DrawingArea, HPacker, VPacker, TextArea
from matplotlib.legend import Legend

In [ ]:
save_result = True
save_fig = save_result
today = datetime.datetime.now().strftime("%Y%m%d")
save_fig_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-detvar-{}".format(today)

In [ ]:
syst_dfs = {}

file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10"
syst_keys = ["CV", "WireMod_XThetaXW", "WireMod_YZ"]
for sidx, syst_key in enumerate(syst_keys):
    filename = "SystVar_{}_wtrks.df".format(syst_key)
    mc_file = path.join(file_dir, filename)
    mc_split_df = pd.read_hdf(mc_file, key="split")
    mc_n_split = get_n_split(mc_file)
    print("mc_n_split: %d" %(mc_n_split))
    print_keys(mc_file)

    n_max_concat = 100
    mc_keys2load = ['hdr', 'evt', 'trk'] 
    mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
    mc_hdr_df = mc_dfs['hdr']
    mc_evt_df = mc_dfs['evt']
    mc_trk_df = mc_dfs['trk']
    mc_trk_df = mc_trk_df[mc_trk_df.pfp.trk.producer != 4294967295]
    mask = (mc_trk_df.pfp.trk.len > 0) &\
         (mc_trk_df.pfp.pfochar.vtxdist < 100) #&\
    mc_trk_df = mc_trk_df[mask]

    nlevels = len(mc_evt_df.columns.levels)
    index_names = mc_evt_df.index.names
    mc_hdr_df.columns = pd.MultiIndex.from_tuples([tuple([str(c)] +[""] * (nlevels-1)) for c in mc_hdr_df.columns]) 
    mc_evt_df = multicol_merge(mc_evt_df.reset_index(), 
                               mc_hdr_df.reset_index(),
                               left_on=["__ntuple", "entry"],
                               right_on=["__ntuple", "entry"],
                               how="left"
                               ) 
    mc_evt_df = mc_evt_df.set_index(index_names, verify_integrity=True) 

    # need to match the nu_Es across files
    mc_evt_df["nu_E"] = mc_evt_df.mc.E

    syst_dfs[syst_key] = mc_evt_df
    syst_dfs[syst_key+"_trk"] = mc_trk_df

del mc_hdr_df
del mc_evt_df

In [ ]:
syst_keys = ["CV", "WireMod_XThetaXW", "WireMod_YZ"]

for k in syst_keys:
    syst_dfs[k] = syst_dfs[k].reset_index().set_index(["run","subrun","evt","nu_E"])
    print(len(syst_dfs[k].index))

for kidx, k in enumerate(syst_keys):
    idxs = syst_dfs[k].index
    if kidx == 0:
        common_idxs = idxs
    else:
        common_idxs = common_idxs.intersection(idxs)
print(len(common_idxs))

for k in syst_keys:
    syst_dfs[k] = syst_dfs[k].loc[common_idxs]

In [ ]:
# total POT (not used for syst sample comparison)
tot_pot = syst_dfs['CV'].groupby(level=[0,1]).nth(0)['pot'].sum()
print("mc_tot_pot: %.3e" %(tot_pot))

pot_str = "{:.2e}".format(tot_pot).replace("e+0", "e").replace("e+", "e")\
    .replace("e", " $\\times 10^{") + "}$"
pot_str = pot_str.replace(".00", "")
pot_str = pot_str.replace("$\\times 10^{", "$\\times 10^{")  # Ensure closing brace is present
pot_str = pot_str.replace("}$", "}")  # Remove extra $ in closing
# For LaTeX formatting, convert '1.23 $\\times 10^{18}' to '1.23 $\\times 10^{18}$'
pot_str = pot_str if pot_str.endswith("$") else pot_str + "$"
pot_str = pot_str.replace(" $", "$")  # Fix any misplaced spaces
print(pot_str)

labels = ["CV", r"X-$\theta_{xw}$", r"Y-Z"]
colors = ["black", "C0", "C1"]

In [ ]:
# events selection cut thresholds
nu_score_th = 0.45

save_ntrks = 2

trackscore_th = 0.5

vtxdist_th = 1.2

mu_chi2mu_th = 30
mu_chi2p_th = 100
mu_len_th = 50

qual_th = 0.2

p_chi2p_th = 90
p_len_th = 0

mu_Plo_th = 0.22
mu_Phi_th = 1
p_Plo_th = 0.3
p_Phi_th = 1

In [ ]:
# sanity check to see if nus are matched correctly
bins = np.linspace(0.13, 4, 100)
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]
vardf = [df[("mc","E")] for df in evtdf]
vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
plot_labels = ["Neutrino Energy", 
        "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("E")
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
bins = np.linspace(0.2, 0.8, 100)
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]
vardf = [df[("slc","nu_score")] for df in evtdf]
# vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
plot_labels = ["nu_score", 
        "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("nu_score")
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = cut_clear_cosmic(syst_dfs[syst_key])
    syst_dfs[syst_key] = cut_vertex_in_fv(syst_dfs[syst_key])
    syst_dfs[syst_key] = cut_nu_score(syst_dfs[syst_key], nu_score_th)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = syst_dfs[syst_key].reset_index().set_index(list(syst_dfs[f'{syst_key}_trk'].index.names)[:-1])
    syst_dfs[syst_key+"_trk"] = get_valid_trks(syst_dfs[syst_key+"_trk"])
    syst_dfs[syst_key+"_trk"] = match_trkdf_to_slcdf(syst_dfs[syst_key+"_trk"], syst_dfs[syst_key])

In [ ]:
bins = np.linspace(0, 10, 100)
evtdf = [syst_dfs[syst_key+"_trk"] for syst_key in syst_keys]
vardf = [df[("pfp","pfochar","vtxdist")] for df in evtdf]
vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
plot_labels = ["vtx_dist", "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("vtx_dist")
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
for k in syst_keys:
    syst_dfs[k] = get_trk_info(syst_dfs[k], syst_dfs[f'{k}_trk'], save_ntrks)
    del syst_dfs[f'{k}_trk']
del mc_trk_df

In [ ]:
bins = np.linspace(1, 8, 8)
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]
vardf = [df[("n_trks")] for df in evtdf]
# vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
save_tag = "n_trks"
plot_labels = [save_tag, 
        "Events (POT={})".format(pot_str), ""]
approval = "preliminary"
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
bins = np.linspace(1, 8, 8)
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]
vardf = [df[("n_good_trks")] for df in evtdf]
# vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
save_tag = "n_good_trks"
plot_labels = [save_tag, 
        "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = cut_2prong_contained(syst_dfs[syst_key])

In [ ]:
bins = np.linspace(0, 1, 51)
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

vardf = [df[("trk1", "pfp", "trackScore")] for df in evtdf]
save_tag = "trk1_trackScore"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

vardf = [df[("trk2", "pfp", "trackScore")] for df in evtdf]
save_tag = "trk2_trackScore"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
approval = "preliminary"
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = cut_2prong_trackscore(syst_dfs[syst_key], trackscore_th)
    syst_dfs[syst_key] = cut_2prong_vtxdist(syst_dfs[syst_key], vtxdist_th)

In [ ]:
syst_dfs["CV"].trk1.pfp.trk.chi2pid.I2

In [ ]:
bins = np.linspace(0, 60, 61)
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

vardf = [df[("trk1", "pfp", "trk", "chi2pid", "I2", "chi2_muon")] for df in evtdf]
save_tag = "trk1_chi2_muon"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

vardf = [df[("trk2", "pfp", "trk", "chi2pid", "I2", "chi2_muon")] for df in evtdf]
save_tag = "trk2_chi2_muon"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
bins = np.linspace(0, 300, 51)
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

vardf = [df[("trk1", "pfp", "trk", "chi2pid", "I2", "chi2_proton")] for df in evtdf]
save_tag = "trk1_chi2_proton"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

vardf = [df[("trk2", "pfp", "trk", "chi2pid", "I2", "chi2_proton")] for df in evtdf]
save_tag = "trk2_chi2_proton"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

bins = np.linspace(0, 400, 51)
vardf = [df[("trk1", "pfp", "trk", "len")] for df in evtdf]
save_tag = "trk1_len"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

bins = np.linspace(0, 150, 51)
vardf = [df[("trk2", "pfp", "trk", "len")] for df in evtdf]
save_tag = "trk2_len"
plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format(save_tag)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = get_mu_p_candidate(syst_dfs[syst_key], 
                            mu_chi2mu_th=mu_chi2mu_th, mu_chi2p_th=mu_chi2p_th, mu_len_th=mu_len_th, qual_th=qual_th,
                            p_chi2mu_th=-1, p_chi2p_th=p_chi2p_th, p_len_th=p_len_th)

    syst_dfs[syst_key] = cut_has_mu(syst_dfs[syst_key]) 
    syst_dfs[syst_key] = cut_mu_kinematics(syst_dfs[syst_key], mu_Plo_th=mu_Plo_th, mu_Phi_th=mu_Phi_th)

    syst_dfs[syst_key] = cut_has_p(syst_dfs[syst_key]) 
    syst_dfs[syst_key] = cut_p_kinematics(syst_dfs[syst_key], p_Plo_th=p_Plo_th, p_Phi_th=p_Phi_th)

In [ ]:
detvar_weights = {}

In [ ]:
for var_config in [VariableConfig.muon_momentum(), VariableConfig.muon_direction(), VariableConfig.proton_momentum(), VariableConfig.proton_direction()]:
        bins = var_config.bins
        evtdf = [syst_dfs[syst_key] for syst_key in syst_dfs.keys()]
        vardf = [df[var_config.var_evt_reco_col[:-1]] for df in evtdf]
        vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
        plot_labels = [var_config.var_labels[1], 
                "Events (POT={})".format(pot_str), ""]
        approval = "preliminary"
        save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
        n = overlay_hists(evtdf, vardf, bins,
                colors, labels,
                plot_labels,
                vline=[],
                save_fig=save_fig, save_name=save_name)
        detvar_weights[var_config.var_save_name] = [n[i]/n[0] for i in range(len(syst_dfs.keys()))]

In [ ]:
tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]
for syst_key in syst_dfs.keys():
    slc_mudf = syst_dfs[syst_key].mu.pfp.trk
    slc_pdf = syst_dfs[syst_key].p.pfp.trk
    slc_P_mu_col = pad_column_name(("P", "p_muon"), slc_mudf)
    slc_P_p_col = pad_column_name(("P", "p_proton"), slc_pdf)
    tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
    for var_name in tki_var_names:
        syst_dfs[syst_key] = multicol_add(syst_dfs[syst_key], tki_reco[var_name].rename(var_name))

In [ ]:
for var_config in [VariableConfig.tki_del_Tp(), VariableConfig.tki_del_Tp_x(), VariableConfig.tki_del_Tp_y(), VariableConfig.tki_del_alpha(), VariableConfig.tki_del_phi()]:
        bins = var_config.bins
        evtdf = [syst_dfs[syst_key] for syst_key in syst_dfs.keys()]
        vardf = [df[var_config.var_evt_reco_col[:-1]] for df in evtdf]
        vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
        plot_labels = [var_config.var_labels[1], 
                "Events (POT={})".format(pot_str), ""]
        approval = "preliminary"
        save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
        n = overlay_hists(evtdf, vardf, bins,
                colors, labels,
                plot_labels,
                vline=[],
                save_fig=save_fig, save_name=save_name)
        detvar_weights[var_config.var_save_name] = [n[i]/n[0] for i in range(len(syst_dfs.keys()))]

# Detector Variation Systematic Uncertainty

In [ ]:
syst_name = "detvar"

In [ ]:
var_config = VariableConfig.muon_momentum()

In [ ]:
bins = var_config.bins
evtdf = [syst_dfs[syst_key] for syst_key in syst_dfs.keys()]
vardf = [df[var_config.var_evt_reco_col[:-1]] for df in evtdf]
vardf = [np.clip(df, bins[0], bins[-1] - eps) for df in vardf]
plot_labels = [var_config.var_labels[1], 
        "Events (POT={})".format(pot_str), ""]
approval = "preliminary"
save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
n = overlay_hists(evtdf, vardf, bins,
        colors, labels,
        plot_labels,
        vline=[],
        save_fig=save_fig, save_name=save_name)
detvar_weights[var_config.var_save_name] = [n[i]/n[0] for i in range(len(syst_dfs.keys()))]

In [ ]:
ret_dict = {}
for kidx, syst_key in enumerate(syst_dfs.keys()):
    if syst_key == "CV":
        continue

    cv_events = n[0]
    univ_events = np.array([n[kidx]])
    ret = get_covariance(univ_events, cv_events)
    ret_dict[syst_key] = ret
    plot_univ_hist(syst_dfs["CV"], var_config, syst_name, univ_events, cv_events)

    matrix_type = "Covariance"
    save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
    title = "{} {}".format(syst_key, matrix_type)
    plot_heatmap(ret[matrix_type], var_config, 
                title=title, 
                save_fig=save_fig, save_fig_name=save_fig_name)

    frac_unc = np.sqrt(np.diag(ret["Covariance_Frac"]))
    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_unc, histtype="step", color="black")
    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.xlabel(var_config.var_labels[0])
    plt.ylabel("Fractional Uncertainty")
    plt.title(syst_key)
    plt.grid(True)
    plt.show();

In [ ]:
# get total detector variation covariance matrix
for kidx, syst_key in enumerate(ret_dict.keys()):
    if kidx == 0:
        detvar_total_cov = ret_dict[syst_key]["Covariance_Frac"]
    else:
        detvar_total_cov += ret_dict[syst_key]["Covariance_Frac"]

frac_unc = np.sqrt(np.diag(detvar_total_cov))
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_unc, histtype="step", color="black")
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Fractional Uncertainty")
plt.title("Detector Variation")
plt.grid(True)
plt.show();

In [ ]:
if 'syst_dict' not in locals():
    syst_dict = {}

syst_dict[var_config.var_save_name] = detvar_total_cov
for syst_key in ret_dict.keys():
    syst_dict[var_config.var_save_name + "_" + syst_key] = ret_dict[syst_key]["Covariance_Frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/detvar_syst_dict.npz", **syst_dict)

# sanity check: load saved file and check it agrees with the local one for all keys
loaded = np.load(file_dir + "/detvar_syst_dict.npz")
for key in syst_dict.keys():
    arr_local = syst_dict[key]
    arr_loaded = loaded[key]
    assert np.allclose(arr_local, arr_loaded), f"Mismatch for key: {key}"
print("All keys in syst_dict match the saved npz file.")